In [ ]:
# Cell 0 — Setup & Installs
# This program sets up the Colab environment with all required libraries.
# Verify CUDA/GPU availability and print versions for reproducibility.
# Unsloth: fast loading + LoRA utilities for small instruct LLMs.
# bitsandbytes: 4-bit/8-bit quantization (saves VRAM).
# Transformers + Accelerate + PEFT + TRL: HF training stack we’ll use.

# Install core libraries
!pip -q install --upgrade pip
!pip -q install "unsloth>=2024.10.0" transformers accelerate peft trl bitsandbytes datasets sentencepiece

# Show attached GPU
!nvidia-smi || true

# Imports & reproducibility
import os, re, glob, math, json, random
import numpy as np
import pandas as pd
from typing import List, Dict, Any

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# PyTorch / CUDA checks
import torch
torch.manual_seed(SEED)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("Device name:", torch.cuda.get_device_name(0))

# HF / TRL / Unsloth entry points (used later)
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

print("\nEnvironment ready. Proceed to Cell 1 to fetch and load the travel dataset from GitHub.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 78.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
Tue Oct 21 03:35:12 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Ut

/tmp/ipython-input-3508429327.py:38: UserWarning: WARNING: Unsloth should be imported before trl, transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

Environment ready. Proceed to Cell 1 to fetch and load the travel dataset from GitHub.


In [ ]:
# Cell 1: Fetch dataset from GitHub + load CSV
# The code automatically download the travel-domain dataset from its GitHub repository.
# Locate the correct CSV file (even if the name or encoding format varies).
# Load it safely into a Pandas DataFrame for further processing.
# This step confirms that the notebook can run on any Colab environment
# without manually uploading files.

# Clone the official dataset repository (clean copy each run)
# Removes any old folder to avoid conflicts, then clones the repo quietly.
!rm -rf travel_domain_data
!git clone -q https://github.com/suralk/travel_domain_data.git

import os, glob, pandas as pd

# Automatically detect the correct CSV file inside the repository.
# Some repos might contain multiple CSVs — we search recursively and pick
# The one that matches known names such as "5000TravelQuestionsDataset.csv".
csv_candidates = glob.glob("travel_domain_data/**/*.csv", recursive=True)
if not csv_candidates:
    raise FileNotFoundError(" No CSV file found in the repository. Please check the repo structure.")

# Common dataset filenames (listed in order of preference)
preferred_names = {
    "5000TravelQuestionsDataset.csv",
    "travel_data.csv",
    "travel_domain_data.csv",
    "travel.csv",
    "data.csv",
    "dataset.csv",
}

# Pick the preferred CSV if found; otherwise, default to the first one.
csv_path = None
for p in csv_candidates:
    if os.path.basename(p) in preferred_names:
        csv_path = p
        break
if csv_path is None:
    csv_path = csv_candidates[0]  # fallback to first detected file

print(" Using CSV file:", csv_path)

# Robust CSV reading procedure
# Some CSVs may have different encodings (UTF-8 vs Latin-1) or broken lines.
# Attempting multiple reading strategies until one works successfully.
read_attempts = [
    dict(encoding="utf-8", engine="c"),
    dict(encoding="ISO-8859-1", engine="c"),
    dict(encoding="utf-8", engine="python", on_bad_lines="skip"),
    dict(encoding="ISO-8859-1", engine="python", on_bad_lines="skip"),
]

last_err = None
df = None
for kw in read_attempts:
    try:
        df = pd.read_csv(csv_path, **kw)
        break  # stop at the first successful read
    except Exception as e:
        last_err = e

if df is None:
    # If all attempts fail, print the last encountered error.
    raise RuntimeError(f"Failed to read CSV with multiple strategies. Last error:\n{last_err}")

# Confirm successful load and preview the data
print(" Dataset loaded successfully!")
print("Total rows:", len(df), "| Columns detected:", list(df.columns))
display(df.head(5))  # showing a sample of the first 5 rows


 Using CSV file: travel_domain_data/5000TravelQuestionsDataset.csv
 Dataset loaded successfully!
Total rows: 4999 | Columns detected: ['What are the special things we (husband and me) can do during a 5 day stay at Cape Town?', 'TTD', 'TTDSIG']


,What are the special things we (husband and me) can do during a 5 day stay at Cape Town?,TTD,TTDSIG
0,What are the companies which organize shark fe...,TTD,TTDOTH
1,Is it safe for female traveller to go alone to...,TGU,TGUHEA
2,What are the best places around Cape Town for ...,TTD,TTDSIG
3,What are the best places to stay for a family ...,ACM,ACMOTH
4,What are the train services that travels from ...,TRS,TRSTRN


In [ ]:
# Cell 2 — Clean dataset & create 4000/300/700 splits
# The program finds the text column and the *coarse* label column (ignore any fine-grain label).
# Keep only {question, label}, clean strings, and map labels to the 7 required classes.
# Shuffle deterministically and split: Train=4000, Val=300, Test=700.

import re, pandas as pd, numpy as np

# 1) Detect the question-text column.
text_candidates = [c for c in df.columns if re.search(r'(question|query|text|title)', c, re.I)]
if not text_candidates:
    # fallback: first string-like column
    text_candidates = [df.select_dtypes(include='object').columns[0]]
TEXT_COL = text_candidates[0]

# 2) Detect the *coarse* label column (avoid any "fine" columns).
#    This strictly follows the assignment requirement to use coarse-grain classes.
coarse_like = [c for c in df.columns if re.search(r'(coarse|coarse_?label)', c, re.I)]
fine_like   = [c for c in df.columns if re.search(r'(fine|fine_?label)',   c, re.I)]

if coarse_like:
    LABEL_COL = coarse_like[0]
else:
    # fallback: generic label/category/topic, but explicitly exclude anything that looks "fine"
    label_candidates = [c for c in df.columns
                        if re.search(r'(label|class|category|topic|type)', c, re.I)
                        and c not in fine_like]
    if not label_candidates:
        # last resort: another object column that isn't the text and not fine-like
        label_candidates = [c for c in df.select_dtypes(include='object').columns
                            if c != TEXT_COL and c not in fine_like][:1]
    if not label_candidates:
        raise ValueError("No suitable COARSE label column found — please inspect df.columns")
    LABEL_COL = label_candidates[0]

print(f"Detected columns →  Question: '{TEXT_COL}' | Coarse Label: '{LABEL_COL}'")

# 3) Keep only the needed columns and clean
df = df[[TEXT_COL, LABEL_COL]].dropna().reset_index(drop=True)
df = df.rename(columns={TEXT_COL: "question", LABEL_COL: "label"})
df["question"] = df["question"].astype(str).str.strip()
df["label"]    = df["label"].astype(str).str.strip()

# 4) Map labels to the 7 coarse categories
valid_labels = {"TTD","TGU","ACM","TRS","WTH","FOD","ENT"}
unique_labels = set(df["label"].unique())
mapping = {}

for lab in unique_labels:
    s = lab.lower()
    if "thing" in s or s == "ttd":
        mapping[lab] = "TTD"
    elif "guide" in s or s == "tgu":
        mapping[lab] = "TGU"
    elif "accom" in s or "hotel" in s or "lodg" in s or s == "acm":
        mapping[lab] = "ACM"
    elif "transport" in s or "bus" in s or "train" in s or s == "trs":
        mapping[lab] = "TRS"
    elif "weather" in s or s == "wth":
        mapping[lab] = "WTH"
    elif "food" in s or "restaurant" in s or s == "fod":
        mapping[lab] = "FOD"
    elif "entertain" in s or "nightlife" in s or s == "ent":
        mapping[lab] = "ENT"
    else:
        mapping[lab] = "TGU"   # safe catch-all → Travel Guide

df["label"] = df["label"].map(mapping)
df = df[df["label"].isin(valid_labels)].reset_index(drop=True)

print("Coarse label mapping complete. Class distribution:")
display(df["label"].value_counts())

# 5) Shuffle deterministically and cap at 5000 if needed
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
if len(df) > 5000:
    df = df.iloc[:5000]

# 6) Split: Train=4000, Val=300, Test=700
N_TRAIN, N_VAL, N_TEST = 4000, 300, 700
train_df = df.iloc[:N_TRAIN].copy()
val_df   = df.iloc[N_TRAIN:N_TRAIN+N_VAL].copy()
test_df  = df.iloc[N_TRAIN+N_VAL:N_TRAIN+N_VAL+N_TEST].copy()

print(f"Final split sizes → Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}")
display(train_df.head(2)); display(val_df.head(2)); display(test_df.head(2))


Detected columns →  Question: 'What are the special things we (husband and me) can do during a 5 day stay at Cape Town?' | Coarse Label: 'TTD'
Coarse label mapping complete. Class distribution:


,count
label,
TGU,1220
TTD,1139
TRS,1011
ACM,720
FOD,521
ENT,216
WTH,172


Final split sizes → Train=4000 | Val=300 | Test=699


,question,label
0,Can anyone suggest a central location to spend...,ACM
1,How many days should I spend in Busan?,TTD


,question,label
4000,What are the day trip destinations in Split?,TTD
4001,What would be a safe place to stay that is nea...,ACM


,question,label
4300,What is the best transportation method from Sa...,TRS
4301,What are the taxi services available for trave...,TRS


In [ ]:
# Cell 3 — Load small instruct model (Unsloth)
# Load a small instruction-tuned LLM using Unsloth’s fast loader.
# The model is loaded in 4-bit precision to save GPU memory (important for Colab).
# Using FP16 (half-precision) for faster computation and lower memory use.
# Once loaded, the model can be used for:
# 1. Inference (zero/few-shot predictions)
# 2. Later fine-tuning using LoRA adapters (lightweight training)

# The Unsloth library automatically optimizes speed and memory.

from unsloth import FastLanguageModel
import torch

# Choose a small "instruct" model (pre-trained to follow text instructions)
# You can try other compatible models by changing the model name below.
# Example alternative: "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MODEL_NAME = "unsloth/llama-3-8b-Instruct-bnb-4bit"

# Configuration:
# USE_4BIT: loads model weights in 4-bit precision (reduces memory by around 75%)
# DTYPE: sets compute type to float16 (half precision) for faster inference
USE_4BIT   = True
DTYPE      = torch.float16

print("Loading model:", MODEL_NAME)

# Load the pre-trained model and its tokenizer
# max_seq_length = 4096: allows long prompts (good for few-shot examples)
# dtype & load_in_4bit control precision and memory usage
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = 4096,      # context window size (max input length)
    dtype           = DTYPE,
    load_in_4bit    = USE_4BIT,  # use 4-bit quantization to save GPU VRAM
)

# Optimize the model for inference (faster generation kernels)
FastLanguageModel.for_inference(model)

# Configure tokenizer behavior:
# padding_side='left': pads shorter inputs on the left for batch generation
# truncation_side='left': trims older context from the left when too long
tokenizer.padding_side    = "left"
tokenizer.truncation_side = "left"

# Print basic info to confirm successful setup
print("Pad token id:", tokenizer.pad_token_id, "| pad token:", repr(tokenizer.pad_token))
print("Model and tokenizer successfully loaded and ready to use!")


Loading model: unsloth/llama-3-8b-Instruct-bnb-4bit
==((====))==  Unsloth 2025.10.7: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Pad token id: 128255 | pad token: '<|reserved_special_token_250|>'
Model and tokenizer successfully loaded and ready to use!


In [ ]:
# Cell 4: Prompt templates & generation params
# Define how the model will be prompted in both zero-shot and few-shot cases.
# These templates guide the LLM to classify each question into one of 7 categories.
# MOreover, set up the text generation parameters that control how random or confident the model’s answers are.

# Code Explanation:
# Zero-shot: The model sees only the question, not any examples.
# Few-shot: The model sees 1 or 3 example Q&A pairs before the real question.
# These prompts are automatically filled with travel-related questions from the dataset.

# List of the 7 coarse labels the model must choose from
LABELS = ["TTD", "TGU", "ACM", "TRS", "WTH", "FOD", "ENT"]

# Zero-shot template
# In zero-shot mode, the model receives only the question and must pick one of the valid labels.
# The curly braces {question} will be replaced by each real question from the dataset.
ZS_TEMPLATE = """You are a classifier for travel-domain questions.
Respond with exactly one label from: TTD, TGU, ACM, TRS, WTH, FOD, ENT.

Labels:
TTD (things to do)
TGU (travel guide)
ACM (accommodation)
TRS (transport)
WTH (weather)
FOD (food)
ENT (entertainment)

Question:
{question}

Answer with only the label (e.g., TTD).
"""

# Function to dynamically create few-shot prompts (1-shot or 3-shot)
def build_fewshot_prompt(question: str, k: int, examples_df: pd.DataFrame) -> str:
    """
    Build few-shot prompts (1-shot or 3-shot) dynamically using examples from the training data.

    Arguments:
        question: The test question to classify.
        k: Number of example pairs to include (1 or 3).
        examples_df: A DataFrame containing example (question, label) pairs.

    Returns:
        A formatted string prompt with k example Q&A pairs followed by the target question.
    """
    assert k in (1, 3), "Few-shot prompt only supports k=1 or k=3."

    # Randomly select k examples from the training dataset (same random seed for reproducibility)
    shots = examples_df.sample(n=min(k, len(examples_df)), random_state=42)

    # Begin constructing the prompt step-by-step
    parts = [
        "You are a classifier for travel-domain questions.",
        "Respond with exactly one label from: TTD, TGU, ACM, TRS, WTH, FOD, ENT.\n"
    ]

    # Add k example pairs (each includes one sample question and its correct label)
    for i, row in enumerate(shots.itertuples(index=False), 1):
        parts += [
            f"Example {i}:",
            f"Question: {row.question}",
            f"Label: {row.label}",
            ""
        ]

    # Add the real test question at the end
    parts += [
        "Now classify the following:",
        f"Question: {question}",
        "Answer with only the label (e.g., TTD)."
    ]

    # Join all the text parts into one full prompt string
    return "\n".join(parts)

# Generation hyperparameters:
# These control how the model generates text — lower randomness → more consistent labels.

TEMP = 0.2      # Temperature: lower = more deterministic (stable answers)
TOP_P = 0.9     # Top-p (nucleus sampling): only consider top 90% probability tokens
MAX_NEW_TOKENS = 8  # Limit model output length (just enough to return a label)

# Print confirmation for logs
print(" Prompt templates successfully created.")
print("Generation parameters → temperature:", TEMP, "| top_p:", TOP_P, "| max_new_tokens:", MAX_NEW_TOKENS)


 Prompt templates successfully created.
Generation parameters → temperature: 0.2 | top_p: 0.9 | max_new_tokens: 8


In [ ]:
# Cell 5: Helper functions and Zero-shot Evaluation
# Define reusable helper functions for model prediction and accuracy calculation.
# Perform zero-shot testing (the model classifies without seeing any examples).
# "Zero-shot" means the model must predict the correct label
# Using only the prompt instructions, not prior examples.
# Accuracy is used to measure how often the model’s predicted label
# matches the correct one from the dataset.

import re, numpy as np, torch
from typing import List

# Helper function 1: classify_batch()
def classify_batch(prompts: List[str]) -> List[str]:
    """
    Given a list of text prompts, generate predictions from the model
    and extract exactly one of the 7 valid labels.

    In other words:
      - Converts text prompts into token IDs that the model can understand.
      - Runs the model to generate responses.
      - Cleans and extracts only the label token (e.g., 'TTD', 'WTH', etc.)
        from the generated output.
    """
    # Tokenize the list of prompts (convert text → tensors for the model)
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,       # pad shorter prompts for batch processing
        truncation=True,    # trim very long text if necessary
    ).to(model.device)

    # Disable gradient tracking (faster inference, less memory)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,   # short output — only label
            do_sample=True,                  # enables stochastic decoding
            temperature=TEMP,                # controls randomness
            top_p=TOP_P,                     # keeps only most likely tokens
            pad_token_id=tokenizer.eos_token_id,
        )

    # Convert generated tokens back into readable text
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    preds = []
    for prompt, full in zip(prompts, decoded):
        # Extract only the generated part after the prompt
        tail = full[len(prompt):] if full.startswith(prompt) else full
        # Find any of the 7 known label tokens
        m = re.search(r"\b(TTD|TGU|ACM|TRS|WTH|FOD|ENT)\b", tail)
        preds.append(m.group(1) if m else "TGU")  # fallback = TGU (safe default)
    return preds


# Helper function 2: accuracy()
def accuracy(y_true: List[str], y_pred: List[str]) -> float:
    """
    Calculate exact-match accuracy between predictions and true labels.

    In other words:
      - Compares each prediction with its correct label.
      - Returns the percentage of correct answers.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return float((y_true == y_pred).mean())

# Zero-shot Evaluation

# Prepare test data lists (questions + true labels)
test_questions = test_df["question"].tolist()
test_labels    = test_df["label"].tolist()

# Create a list of zero-shot prompts using the template from Cell 4
zs_prompts = [ZS_TEMPLATE.format(question=q) for q in test_questions]

# Batch the evaluation to avoid GPU memory overload
zs_pred = []
BATCH = 16  # 16 prompts per batch (safe for Colab GPU)
for i in range(0, len(zs_prompts), BATCH):
    zs_pred.extend(classify_batch(zs_prompts[i:i+BATCH]))

# Calculate accuracy on the test set
zs_acc = accuracy(test_labels, zs_pred) if len(test_labels) == len(zs_pred) else float('nan')

# Print the final zero-shot performance result
print(f" Zero-shot accuracy: {zs_acc:.4f}")


 Zero-shot accuracy: 0.3720


In [ ]:
# Cell 6: 1-shot and 3-shot evaluation
# The code tests how few-shot prompting improves classification accuracy.
# Compare the model’s performance when given 1 example (1-shot)
# or 3 examples (3-shot) before answering each test question.

# Basically, in “few-shot learning,” the model is shown a few examples of the task
# inside the prompt before being asked to classify a new question.
# Using the helper functions from previous cells:
# 1. Build_fewshot_prompt() from Cell 4
# 2. Classify_batch() and accuracy() from Cell 5

import pandas as pd

# Function: Evaluate model in k-shot setting (k = 1 or 3)
def eval_k_shot(k: int, train_df: pd.DataFrame, test_df: pd.DataFrame) -> float:
    """
    Evaluate few-shot prompting with k examples (where k = 1 or 3).

    In other words:
      - For each test question, build a new prompt that includes k examples
        randomly chosen from the training data.
      - The model then predicts the correct label for that test question.
      - Finally, compute how accurate those predictions are.

    Args:
        k (int): number of example pairs (1 or 3)
        train_df (pd.DataFrame): training data used to sample examples
        test_df (pd.DataFrame): test data to evaluate accuracy on
    Returns:
        float: accuracy score for this few-shot setting
    """
    assert k in (1, 3), "k must be 1 or 3"
    preds, golds = [], []

    # Loop through each test question and classify
    for row in test_df.itertuples(index=False):
        # Build a few-shot prompt using k training examples
        prompt = build_fewshot_prompt(row.question, k=k, examples_df=train_df[["question", "label"]])
        # Get the model’s prediction for this single prompt
        pred = classify_batch([prompt])[0]
        preds.append(pred)
        golds.append(row.label)

    # Calculate accuracy across all test questions
    return accuracy(golds, preds)

#Run both few-shot experiments
one_shot_acc   = eval_k_shot(1, train_df, test_df)
three_shot_acc = eval_k_shot(3, train_df, test_df)

# Display the results clearly
print(f" 1-shot accuracy : {one_shot_acc:.4f}")
print(f" 3-shot accuracy : {three_shot_acc:.4f}")


 1-shot accuracy : 0.2532
 3-shot accuracy : 0.2203


In [ ]:
# Cell 7: Prepare SFT data (chat format) + LoRA config

# This code prepares the dataset for supervised fine-tuning (SFT) by converting
# each (question, label) pair into a chat-style format that LLMs understand.
# Attach lightweight LoRA adapters to the model for efficient training.

# During SFT, the model learns from examples where:
# 1. "system" gives the overall instruction
# 2. "user" provides the question
# 3. "assistant" provides the correct label
# LoRA adapters allow fine-tuning only small parts of the model (saves GPU memory).

from typing import List, Dict, Any
from unsloth import FastLanguageModel

# Step 1: Convert dataset into chat format
def to_chat_format(df_in: pd.DataFrame) -> List[Dict[str, Any]]:
    """
    Convert (question, label) pairs into a chat-style supervised dataset.

    In other words:
      - Each training example becomes a short conversation.
      - The model sees a system instruction (task definition),
        then a user question, and finally the correct answer (label).

    Example structure:
        system:    "You are a classifier for travel-domain questions..."
        user:      "Question: What is the best hotel in Auckland?"
        assistant: "ACM"
    """
    chats = []
    # System instruction defines the classification task and possible outputs
    system_msg = (
        "You are a classifier for travel-domain questions. "
        "Respond with exactly one label from: TTD, TGU, ACM, TRS, WTH, FOD, ENT. "
        "Answer ONLY the label."
    )

    # Build a list of chat-style training examples
    for row in df_in.itertuples(index=False):
        chats.append({
            "messages": [
                {"role": "system", "content": system_msg},
                {"role": "user",   "content": f"Question: {row.question}"},
                {"role": "assistant", "content": row.label},
            ]
        })
    return chats

# Convert training and validation data into chat format
train_chats = to_chat_format(train_df)
val_chats   = to_chat_format(val_df)
print("Prepared chat-format data → train:", len(train_chats), "| val:", len(val_chats))

# Step 2: Configure LoRA (Low-Rank Adaptation)
# LoRA adds small "adapter" layers to the model so we fine-tune only a few parameters.
# This makes training much faster and memory-efficient — ideal for Colab GPUs.

lora_r        = 16         # rank of the low-rank adapter (as required in assignment)
lora_alpha    = 32         # scaling factor for adapter weights
lora_dropout  = 0.05       # dropout rate (prevents overfitting)

# Specify which parts of the model LoRA will update:
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",   # attention mechanism projections
    "gate_proj", "up_proj", "down_proj"       # MLP (feed-forward) layers
]

# Step 3: Attach LoRA adapters to the model
# This modifies the model so that only the selected layers are trainable.
# Gradient checkpointing reduces GPU memory use during backpropagation.

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_r,
    lora_alpha = lora_alpha,
    lora_dropout = lora_dropout,
    target_modules = target_modules,
    bias = "none",
    use_gradient_checkpointing = "unsloth",   # saves GPU memory
    random_state = 42,
)

print("LoRA adapters attached (r=16). Model ready for tokenization and fine-tuning.")


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Prepared chat-format data → train: 4000 | val: 300


Unsloth 2025.10.7 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


LoRA adapters attached (r=16). Model ready for tokenization and fine-tuning.


In [ ]:
# Cell 8: Tokenize chat data for SFT training
# This code turns our chat-style examples into model-readable text tokens.
# Apply the chat template to create a single text conversation for each example.
# Tokenize (convert text → numbers) with padding/truncation so all sequences are the same length.
# Create the labels for supervised fine-tuning (SFT) — in causal language models,
# the model tries to predict the next token, so labels = input_ids.
# This step transforms the data from a human-readable chat format
# into the numeric format required for training the model efficiently.

from datasets import Dataset

# Step 1: Apply chat template to each example
def format_messages(example):
    """
    Combine the list of 'messages' (system, user, assistant)
    into one continuous text string using the tokenizer’s chat template.

    In other words:
      - The chat template converts structured roles into plain text
        in the same format used when training LLMs.
      - Example:
            System: You are a classifier...
            User: Question: What is the best restaurant?
            Assistant: FOD
    """
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,             # return plain text, not tokens yet
            add_generation_prompt=False # no extra "assistant:" prompt added
        )
    }

# Step 2: Build Hugging Face datasets
# Create Hugging Face Dataset objects directly from our list of chat dictionaries.
# The map() function applies format_messages() to every item and removes the old 'messages' column.
train_ds = Dataset.from_list(train_chats).map(format_messages, remove_columns=["messages"])
val_ds   = Dataset.from_list(val_chats).map(format_messages, remove_columns=["messages"])

# Step 3: Define tokenizer function
def tok(batch):
    """
    Tokenize each chat into model input IDs.
    - truncation=True: cuts text that exceeds 512 tokens.
    - padding='max_length': pads shorter texts to the same length.
    """
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512,   # short max length (512) is enough for Q&A labels
    )

# Step 4: Tokenize the entire training & validation datasets
# --------------------------------------------------
train_tok = train_ds.map(tok, batched=True, remove_columns=["text"])
val_tok   = val_ds.map(tok,   batched=True, remove_columns=["text"])

# Step 5: Add 'labels' for supervised fine-tuning
# In causal language modeling, the goal is to predict the next token,
# so the labels are simply the same as the input token IDs.
train_tok = train_tok.add_column("labels", train_tok["input_ids"])
val_tok   = val_tok.add_column("labels",   val_tok["input_ids"])

# Step 6: Print summaries to confirm dataset shape and structure
print(train_tok)
print(val_tok)


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 300
})


In [ ]:
# Cell 9: Make sure attention_mask exists, then train SFT

# Some tokenizers don't automatically create `attention_mask`. If it's missing,
# we create it so the Trainer knows which tokens are padding vs real content.
# It also guarantee there is a valid pad token ID (fallback to EOS if needed).
# Then we configure the training loop (LoRA SFT) and run it safely.

# Without a correct attention mask, the model could "learn" from padded tokens,
# which hurts training and may crash some collators.
# Having an explicit pad token ID makes batching consistent and avoids runtime errors.

import torch
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

# 1) Make sure the tokenizer has a pad token and pad_token_id
# If a pad token isn't defined, we use the EOS token as padding.
# This is a common, safe fallback for decoder-only LLMs.
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("Set tokenizer.pad_token ->", repr(tokenizer.pad_token), "| id:", tokenizer.pad_token_id)

PAD_ID = tokenizer.pad_token_id  # cache for mask building

# 2) Confirms if attention_mask exists on the tokenized datasets
# If missing, it will build a mask where 1 = real token and 0 = padding token.
def add_attention_mask_if_missing(ds):
    if "attention_mask" not in ds.column_names:
        print("Adding attention_mask to dataset...")
        masks = []
        for ids in ds["input_ids"]:
            masks.append([0 if (tok == PAD_ID) else 1 for tok in ids])
        ds = ds.add_column("attention_mask", masks)
    return ds

train_tok = add_attention_mask_if_missing(train_tok)
val_tok   = add_attention_mask_if_missing(val_tok)

# 3) Define training hyperparameters (LoRA SFT)
# Colab-friendly settings:
# 1 epoch to keep time reasonable (increase if you have a better GPU)
# LR = 2e-5 works well for adapter-only tuning
# Cosine scheduler + 3% warmup improves stability in FP16
EPOCHS        = 1
LR            = 2e-5
WEIGHT_DECAY  = 0.0
WARMUP_RATIO  = 0.03
BATCH_SIZE    = 4
GRAD_ACC      = 4  # simulates effective batch size of 16

training_args = TrainingArguments(
    output_dir                  = "outputs",
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACC,
    learning_rate               = LR,
    weight_decay                = WEIGHT_DECAY,
    warmup_ratio                = WARMUP_RATIO,
    lr_scheduler_type           = "cosine",
    logging_steps               = 25,
    eval_strategy               = "epoch",   # compatible arg name across versions
    save_strategy               = "no",      # skip checkpoints to save disk/time
    fp16                        = True,      # half-precision compute on most Colab GPUs
    bf16                        = False,
    dataloader_num_workers      = 2,
    report_to                   = "none",    # disable external logging
)

# 4) Robust data collator
# Builds a batch of tensors and (re)creates attention_mask if an item lacks it.
def data_collator(features):
    input_ids = [f["input_ids"] for f in features]
    if "attention_mask" in features[0]:
        attention_mask = [f["attention_mask"] for f in features]
    else:
        attention_mask = [[0 if tok == PAD_ID else 1 for tok in ids] for ids in input_ids]
    labels = [f["labels"] for f in features]
    return {
        "input_ids":      torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels":         torch.tensor(labels, dtype=torch.long),
    }

# 5) Build Trainer and run SFT
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_tok,
    eval_dataset  = val_tok,
    args          = training_args,
    data_collator = data_collator,
)

trainer.train()
model.eval()

# Switch back to fast generation kernels for inference after training
FastLanguageModel.for_inference(model)
print("SFT training complete (attention_mask verified and applied).")


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,000 | Num Epochs = 1 | Total steps = 250
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss


In [ ]:
# Cell 10: Evaluate SFT on the test set
# This code checks how well the fine-tuned (SFT) model performs on unseen test data.
# keeping the exact same zero-shot prompt format to make the comparison fair
# (so differences come from the model update, not a different prompt).

# Steps:
#   1) Build one prompt per test question using the zero-shot template.
#   2) Run the model in batches to get predictions (avoid GPU OOM).
#   3) Compare predictions with ground-truth labels and print accuracy.

# 1) Build zero-shot-style prompts for every test question (fair apples-to-apples)
sft_prompts = [ZS_TEMPLATE.format(question=q) for q in test_df["question"].tolist()]

# 2) Run batched inference (keeps memory usage stable on Colab GPUs)
sft_pred = []
BATCH = 16
for i in range(0, len(sft_prompts), BATCH):
    sft_pred.extend(classify_batch(sft_prompts[i:i+BATCH]))

# 3) Compute accuracy against the true labels
test_labels = test_df["label"].tolist()
sft_acc = accuracy(test_labels, sft_pred) if len(test_labels) == len(sft_pred) else float('nan')

print(f" SFT accuracy: {sft_acc:.4f}")


 SFT accuracy: 0.3462


In [ ]:
# Cell 11: Results comparison
# This code brings all final accuracy scores together from the experiments.
# Compare Zero-shot, 1-shot, 3-shot, and SFT (fine-tuned) performance.
# This summary table clearly show the accuracy results

import pandas as pd

# Make sure all accuracy variables exist from previous cells:
#   zs_acc (zero-shot)
#   one_shot_acc (1-shot)
#   three_shot_acc (3-shot)
#   sft_acc (supervised fine-tuned)
results = pd.DataFrame({
    "Experiment": ["Zero-shot", "1-shot", "3-shot", "SFT"],
    "Accuracy":   [float(zs_acc), float(one_shot_acc), float(three_shot_acc), float(sft_acc)],
})

# Display results nicely as a summary table
display(results)



,Experiment,Accuracy
0,Zero-shot,0.371960
1,1-shot,0.253219
2,3-shot,0.220315
3,SFT,0.346209


In [ ]:
# Cell A — Model metadata for the report (Q2)

# This cell extracts and displays detailed model information
# It is required for Question 2 of the assignment report.
# It helps justify model selection (e.g., architecture, tokenizer type,
# activation function, embedding design, etc.).
# Also prints important configuration details automatically from the model.

import json
from pprint import pprint

# 1) Tokenizer and model overview
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Vocab size:", getattr(tokenizer, "vocab_size", None))
print("Model class:", model.__class__.__name__)

# 2) Extract key configuration fields directly from model.config
cfg = getattr(model, "config", None)
if cfg is not None:
    # Collect commonly reported architectural parameters.
    meta = {
        "model_type": getattr(cfg, "model_type", None),                     # e.g., 'llama'
        "hidden_size": getattr(cfg, "hidden_size", None),                   # dimension of hidden states
        "intermediate_size": getattr(cfg, "intermediate_size", None),       # MLP expansion dimension
        "num_hidden_layers": getattr(cfg, "num_hidden_layers", None),       # transformer block count
        "num_attention_heads": getattr(cfg, "num_attention_heads", None),   # attention heads per layer
        "max_position_embeddings": getattr(cfg, "max_position_embeddings", None),
        "rope_theta": getattr(cfg, "rope_theta", None),                     # rotary embedding base frequency
        "rms_norm_eps": getattr(cfg, "rms_norm_eps", None),                 # epsilon for normalization stability
        "hidden_act": getattr(cfg, "hidden_act", None),                     # usually 'silu' → SwiGLU-type activation
        "pad_token_id": getattr(cfg, "pad_token_id", None),
        "bos_token_id": getattr(cfg, "bos_token_id", None),
        "eos_token_id": getattr(cfg, "eos_token_id", None),
        "tie_word_embeddings": getattr(cfg, "tie_word_embeddings", None),   # whether input/output embeddings are shared
    }
    pprint(meta)
else:
    print("No model.config available — unable to extract metadata.")

# Print quick reminders for report writing (Q2 context)
print("\nContext length used in this notebook:", 4096)
print("Generation parameters (from Cell 4): temperature =", TEMP,
      ", top_p =", TOP_P, ", max_new_tokens =", MAX_NEW_TOKENS)
print("SFT/LoRA training details (Cells 8–10): r=16, 4-bit base, fp16=True, cosine scheduler, warmup_ratio=0.03, weight_decay=0.0")



Tokenizer class: PreTrainedTokenizerFast
Vocab size: 128000
Model class: PeftModelForCausalLM
{'bos_token_id': 128000,
 'eos_token_id': 128009,
 'hidden_act': 'silu',
 'hidden_size': 4096,
 'intermediate_size': 14336,
 'max_position_embeddings': 8192,
 'model_type': 'llama',
 'num_attention_heads': 32,
 'num_hidden_layers': 32,
 'pad_token_id': 128255,
 'rms_norm_eps': 1e-05,
 'rope_theta': 500000.0,
 'tie_word_embeddings': False}

Context length used in this notebook: 4096
Generation parameters (from Cell 4): temperature = 0.2 , top_p = 0.9 , max_new_tokens = 8
SFT/LoRA training details (Cells 8–10): r=16, 4-bit base, fp16=True, cosine scheduler, warmup_ratio=0.03, weight_decay=0.0


In [ ]:
# Cell B — Qualitative samples for Zero/1/3-shot and SFT

# This cell shows actual model predictions side-by-side across
# different evaluation settings (Zero-shot, 1-shot, 3-shot, and SFT).
# Helps visually identify what kinds of questions each version
# of the model gets right or wrong.

import pandas as pd

# Step 1: Select a small, readable subset for inspection
# Instead of printing all 700 test samples, we only inspect the first 30
# for a clear side-by-side comparison.
SAMPLE_N = 30
sample = test_df.head(SAMPLE_N).copy()

# Step 2: Build prompts for each evaluation mode
# Zero-shot: model classifies without seeing examples
# 1-shot / 3-shot: model sees 1 or 3 training examples inside the prompt
zs_prompts    = [ZS_TEMPLATE.format(question=q) for q in sample["question"]]
one_prompts   = [build_fewshot_prompt(q, k=1, examples_df=train_df[["question", "label"]]) for q in sample["question"]]
three_prompts = [build_fewshot_prompt(q, k=3, examples_df=train_df[["question", "label"]]) for q in sample["question"]]

# Step 3: Generate predictions using classify_batch()
# Run each set of prompts through the model.
# Each call processes several items at once for efficiency.
zs_pred     = classify_batch(zs_prompts)
one_pred    = classify_batch(one_prompts)
three_pred  = classify_batch(three_prompts)

# Reuse zero-shot template for SFT-fine-tuned model (same format, fair comparison)
sft_prompts = [ZS_TEMPLATE.format(question=q) for q in sample["question"]]
sft_pred    = classify_batch(sft_prompts)

# Step 4: Combine everything into one comparison table
qual = pd.DataFrame({
    "question":  sample["question"],
    "gold":      sample["label"],         # true label
    "zero_shot": zs_pred,
    "one_shot":  one_pred,
    "three_shot": three_pred,
    "sft":       sft_pred,
})

# Display the first 30 rows so you can manually examine outputs
display(qual.head(30))

# Step 5: Compute quick sample-level accuracies
def acc(col):
    return (qual[col] == qual["gold"]).mean()

print("\nAccuracy on this 30-sample subset:")
print("  Zero-shot :", f"{acc('zero_shot'):.3f}")
print("  1-shot    :", f"{acc('one_shot'):.3f}")
print("  3-shot    :", f"{acc('three_shot'):.3f}")
print("  SFT       :", f"{acc('sft'):.3f}")



,question,gold,zero_shot,one_shot,three_shot,sft
4300,What is the best transportation method from Sa...,TRS,TTD,TTD,TTD,TTD
4301,What are the taxi services available for trave...,TRS,FOD,TGU,TTD,TGU
4302,What are the restaurants near airport of Samoa?,FOD,FOD,TGU,TTD,TGU
4303,Where can I experience authentic thai food?,FOD,TGU,TGU,TTD,TGU
4304,If you can only stop at one glacier which woul...,TTD,TGU,ACM,TTD,TGU
4305,What are the good Norwegian restaurants with t...,FOD,TGU,FOD,TTD,FOD
4306,Where can I get a sand art done in Dubai?,TGU,TGU,TGU,TTD,TGU
4307,Are there any decent gay bars/clubs?,FOD,ENT,ENT,TTD,ENT
4308,Is there ample money changing facilities in si...,TGU,TGU,ACM,TTD,FOD
4309,How is the weather in Agadir on September?,WTH,WTH,TGU,TTD,WTH



Accuracy on this 30-sample subset:
  Zero-shot : 0.300
  1-shot    : 0.367
  3-shot    : 0.100
  SFT       : 0.300


In [ ]:
# Cell C: Echo key hyperparameters to support Q4 / Q6 / Q7

# This cell neatly prints all the key hyperparameters you used in this assignment.
# It supports Questions 4, 6, and 7 in the assignment report:
# Q4 : Decoding settings (temperature, top_p, max_new_tokens)
# Q6 : LoRA and precision setup
# Q7 : Optimizer and scheduler choices

# Q4 — Prompt and Decoding Hyperparameters
print("=== Prompt / Decoding (Q4) ===")
print("temperature:", TEMP, "→ Controls randomness in generation (lower = more stable).")
print("top_p:", TOP_P, "→ Nucleus sampling threshold for token selection.")
print("max_new_tokens:", MAX_NEW_TOKENS, "→ Limits how long the model’s response can be.")

# Q6 — LoRA and Precision Settings for SFT
print("\n=== LoRA / Precision (Q6) ===")
print("use_4bit:", True, "(Model loaded in 4-bit quantization for memory efficiency.)")
print("fp16:", True, "(Half-precision training for speed and lower GPU usage.)")
print("lora_r:", 16, "→ Rank size of LoRA adaptation matrices (balance between capacity and efficiency).")
print("target_modules:", [
    "q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"
], "→ These layers are adapted during fine-tuning.")

# Q7 — Optimizer, Scheduler, and Training Strategy
print("\n=== Optimizer & Scheduler (Q7) ===")
print("optimizer:", "AdamW (Transformers default in Trainer/SFTTrainer)")
print("learning_rate:", 2e-5, "→ Common safe LR for adapter-level fine-tuning.")
print("weight_decay:", 0.0, "→ No weight decay for simplicity (avoids over-regularization).")
print("lr_scheduler_type:", "cosine", "→ Smooth LR decay pattern over time.")
print("warmup_ratio:", 0.03, "→ Gradually increase LR at start to stabilize training.")
print("gradient_accumulation_steps:", 4, "→ Simulates larger batch size without OOM errors.")
print("per_device_batch_size:", 4, "→ Real batch size per GPU device.")


=== Prompt / Decoding (Q4) ===
temperature: 0.2 → Controls randomness in generation (lower = more stable).
top_p: 0.9 → Nucleus sampling threshold for token selection.
max_new_tokens: 8 → Limits how long the model’s response can be.

=== LoRA / Precision (Q6) ===
use_4bit: True (Model loaded in 4-bit quantization for memory efficiency.)
fp16: True (Half-precision training for speed and lower GPU usage.)
lora_r: 16 → Rank size of LoRA adaptation matrices (balance between capacity and efficiency).
target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'] → These layers are adapted during fine-tuning.

=== Optimizer & Scheduler (Q7) ===
optimizer: AdamW (Transformers default in Trainer/SFTTrainer)
learning_rate: 2e-05 → Common safe LR for adapter-level fine-tuning.
weight_decay: 0.0 → No weight decay for simplicity (avoids over-regularization).
lr_scheduler_type: cosine → Smooth LR decay pattern over time.
warmup_ratio: 0.03 → Gradually increase LR at s